<a href="https://colab.research.google.com/github/Yago-Coqueiro/Pydantic-FastCamp/blob/main/Pydantic_pratica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Instalação e Configuração Inicial :

In [2]:
# Instala as dependências necessárias para rodar FastAPI e validações de e-mail
!pip install fastapi httpx email-validator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 9.1 MB/s eta 0:00:00


### Importações e Definição de Tipos :

In [14]:
import enum
from datetime import datetime
from typing import Optional, Any, Self
from pydantic import BaseModel, Field, EmailStr, SecretStr, field_validator, model_validator

# Enumeração para classificação de risco
class Priority(enum.Enum):
    LOW = "Verde"
    MEDIUM = "Amarelo"
    HIGH = "Vermelho"
    EMERGENCY = "Emergência"

class PatientTriagem(BaseModel):
    # Campo sensível oculto em logs para privacidade do paciente, se torna imutável após criação
    cpf: SecretStr = Field(..., description="CPF do paciente", frozen=True)

    name: str = Field(..., min_length=3, examples=["João Silva"])
    email: EmailStr = Field(description="Contato para envio de resultados")

    # Validação de sinais vitais com limites biológicos estritos
    temperature: float = Field(..., ge=34.0, le=42.5, description="Temperatura em Celsius")
    heart_rate: int = Field(..., ge=30, le=250, description="Batimentos por minuto (BPM)")

    # Define a data da triagem automaticamente no momento da criação
    timestamp: datetime = Field(default_factory=datetime.now)

    # Nível de prioridade com valor padrão
    priority: Priority = Priority.LOW

    # Impede erros de temperatura
    @field_validator("temperature")
    @classmethod
    def check_fever_limit(cls, v: float) -> float:
        if v > 41.0:
            # Gera um alerta imediato
            print(f"ALERTA: Temperatura crítica detectada ({v}°C)!")
        return v

    # Cruza dados para sugerir prioridade automática
    @model_validator(mode="after")
    def auto_classify_priority(self) -> Self:

        # Sobe a prioridade se febre alta ou batimento acelerado
        if self.temperature >= 38.5 or self.heart_rate > 120:
            self.priority = Priority.HIGH
        return self

### Implementação da API :

In [13]:
from fastapi import FastAPI
from fastapi.responses import JSONResponse

# Inicializa o framework FastAPI para gerenciar as rotas e a documentação automática
app = FastAPI(title="Sistema de Triagem Digital - Saúde")

# Banco de dados temporário em memória
db_triagem = []

# Recebe os dados, valida e retorna o objeto criado
@app.post("/triagem", response_model=PatientTriagem)
async def registrar_triagem(dados: PatientTriagem):

    # Simula o salvamento no prontuário eletrônico
    db_triagem.append(dados)
    return dados

# Recupera todos os registros de triagem realizados durante a sessão
@app.get("/pacientes", response_model=list[PatientTriagem])
async def listar_pacientes():
    return db_triagem

### Testes Automatizados :

In [16]:
from fastapi.testclient import TestClient

# Cria um cliente de teste sem subir um servidor real
client = TestClient(app)

def rodar_testes_clinicos():
    print("Teste 1: Paciente com Febre (Deve ser classificado como HIGH)")

    # Dicionário simulando uma entrada de dados com temperatura elevada (39.5)
    data_febre = {
        "cpf": "123.456.789-00",
        "name": "Maria Oliveira",
        "email": "maria@email.com",
        "temperature": 39.5,
        "heart_rate": 85
    }

    # Simula uma requisição POST
    res = client.post("/triagem", json=data_febre)

    # Exibe o status da resposta e o valor da prioridade processada pela lógica do modelo
    print(f"Status: {res.status_code} | Prioridade Gerada: {res.json()['priority']}")

    print("\nTeste 2: Temperatura Impossível (Deve ser bloqueado pelo Pydantic)")
    # Dados que violam as restrições biológicas definidas
    data_erro = {
        "cpf": "000",
        "name": "Erro",
        "email": "email-invalido",
        "temperature": 55.0,      # Valor errado
        "heart_rate": 70
    }

    # Dispara a requisição que deve ser interceptada
    res_erro = client.post("/triagem", json=data_erro)

    # O código 422 indica erros de validação
    print(f"Status esperado (422): {res_erro.status_code}")
    if res_erro.status_code == 422:
        print("Sucesso: O sistema impediu a entrada de um valor biológico impossível.")

# Execução da função de teste se o script for chamado diretamente
if __name__ == "__main__":
    rodar_testes_clinicos()

Teste 1: Paciente com Febre (Deve ser classificado como HIGH)
Status: 200 | Prioridade Gerada: Vermelho

Teste 2: Temperatura Impossível (Deve ser bloqueado pelo Pydantic)
Status esperado (422): 422
Sucesso: O sistema impediu a entrada de um valor biológico impossível.


### Testes de Robustez :

In [18]:
def rodar_testes():
    print("Teste 3: Limite Mínimo Biológico (Deve ser aceito)")
    # Testando o valor exato do limite de temperatura (34)
    data_limite = {
        "cpf": "111.222.333-44",
        "name": "Paciente Limite",
        "email": "limite@saude.com",
        "temperature": 34.0,
        "heart_rate": 60
    }
    res_limite = client.post("/triagem", json=data_limite)
    print(f"Status 34.0°C (Esperado 200): {res_limite.status_code}")

    print("\nTeste 4: Nome Muito Curto (Deve falhar)")
    # Campo name com 2 caracteres (minimo 3).
    data_nome_curto = {
        "cpf": "555.555.555-55",
        "name": "Ab",
        "email": "teste@saude.com",
        "temperature": 36.5,
        "heart_rate": 72
    }
    res_nome = client.post("/triagem", json=data_nome_curto)
    print(f"Status Nome Curto (Esperado 422): {res_nome.status_code}")

    print("\nTeste 5: Formato de E-mail Inválido (Deve falhar)")
    # O tipo EmailStr do Pydantic valida a estrutura do endereço
    data_email_ruim = {
        "cpf": "999.999.999-99",
        "name": "Jose Silva",
        "email": "jose_at_gmail.com", # Falta o símbolo @
        "temperature": 37.0,
        "heart_rate": 80
    }
    res_email = client.post("/triagem", json=data_email_ruim)
    print(f"Status E-mail Inválido (Esperado 422): {res_email.status_code}")

    print("\nTeste 6: Persistência de Dados (Verificação de listagem)")

    # Garante que os pacientes válidos dos testes anteriores foram salvos
    res_lista = client.get("/pacientes")
    quantidade = len(res_lista.json())
    print(f"Total de pacientes triados com sucesso: {quantidade}")

    # Se os testes 1 e 3 passaram, o total deve ser pelo menos 2
    assert quantidade >= 2

# Executa os testes
if __name__ == "__main__":
    rodar_testes()

Teste 3: Limite Mínimo Biológico (Deve ser aceito)
Status 34.0°C (Esperado 200): 200

Teste 4: Nome Muito Curto (Deve falhar)
Status Nome Curto (Esperado 422): 422

Teste 5: Formato de E-mail Inválido (Deve falhar)
Status E-mail Inválido (Esperado 422): 422

Teste 6: Persistência de Dados (Verificação de listagem)
Total de pacientes triados com sucesso: 4
